# Fluid Surrogate — scaling to 1B parameters (Colab)

Two paths, one notebook:

- **Path A — scale from scratch.** Scaled UNet / DiT surrogate on higher-resolution
  fluids (256² Navier–Stokes / Burgers). ~10M on a T4, ~100M on an A100.
  1B from scratch needs an A100-80GB or G4-96GB (see VRAM table).
- **Path B — 1B backbone + LoRA.** Take a pretrained ~1B generative backbone
  (video/DiT) and fine-tune an action-conditioned adapter with LoRA/QLoRA.
  This fits a T4/A100 and is the recommended 1B route.

| setup | bytes/param | 1B params |
|---|---|---|
| infer fp16/bf16 | 2 | ~2 GB |
| train Adam fp32 | ~12 | ~12 GB + activations |
| train bf16 + 8-bit Adam + checkpointing | ~4–6 | ~4–6 GB + activations |

Setup: `Runtime → Change runtime type → GPU` (T4 free; A100/G4 on Pro/Pro+).
Then `Runtime → Run all`. Checkpoints + results persist to Google Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, psutil, shutil
print('cuda:', torch.cuda.is_available(), '| gpus:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram_GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('ram_GB:', round(psutil.virtual_memory().total / 1e9, 1))
print('disk_free_GB:', round(shutil.disk_usage('/content').free / 1e9, 1))
print('torch:', torch.__version__, '| bf16_supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

In [ ]:
!pip install -q bitsandbytes peft einops h5py psutil 2>&1 | tail -2
import bitsandbytes, peft
print('bnb + peft ok')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
ROOT = '/content/drive/MyDrive/virtual-labs/Fluid Surrogate'  # copy this folder there
assert os.path.isdir(ROOT + '/virtual_lab'), 'Upload Fluid Surrogate/ to Drive MyDrive/virtual-labs/'
sys.path.insert(0, ROOT)
os.chdir(ROOT)
import virtual_lab
print('virtual_lab ok ->', virtual_lab.__file__)
CKPT = ROOT + '/checkpoints_1B'; os.makedirs(CKPT, exist_ok=True)
DATA = ROOT + '/data_256'; os.makedirs(DATA, exist_ok=True)
print('ckpt:', CKPT, '| data:', DATA)

## Data — scale resolution and action coverage, not just params

1B params on 10k pairs of 64×64 Burgers would just memorize. Sharded float32
generation at 256² with broad `do(A)` coverage (force / viscosity / obstacles).
Shards keep RAM flat; the loader memory-maps them.

In [ ]:
import numpy as np
from virtual_lab.oracle import BurgersOracle
from virtual_lab.data_gen import random_action, action_to_vec

def gen_shard(path, trajs=50, steps=40, seed=0, n=256, obstacle_p=0.1, dtype=np.float32):
    rng = np.random.default_rng(seed)
    Yt, At, Yn, C = [], [], [], []
    for tr in range(trajs):
        orc = BurgersOracle(n=n, seed=seed + tr)
        orc.reset(nu=float(rng.uniform(0.001, 0.05)))
        for _ in range(steps):
            s = orc.get_state()
            a = random_action(rng, obstacle_p, n)
            s2 = orc.step(a)
            Yt.append(np.stack([s['u'], s['v']]).astype(dtype))
            At.append(action_to_vec(a))
            Yn.append(np.stack([s2['u'], s2['v']]).astype(dtype))
            C.append(s['C']['nu'])
    np.savez_compressed(path, Y_t=np.array(Yt), A_t=np.array(At, dtype=np.float32),
                        Y_next=np.array(Yn), C=np.array(C, dtype=np.float32))
    print(f'saved {path}: {np.array(Yt).shape}')

N_SHARDS, N_GRID = 8, 256  # 8 x (50x40) = 16k pairs @256^2. Raise on A100/G4.
for i in range(N_SHARDS):
    p = f'{DATA}/shard_{i:02d}.npz'
    if not os.path.exists(p):
        gen_shard(p, seed=1000 + i, n=N_GRID)
print('data ready:', sorted(os.listdir(DATA)))

In [ ]:
import torch
from torch.utils.data import Dataset

class ShardDataset(Dataset):
    def __init__(self, paths):
        self.idx, self.cache = [], {}
        for p in paths:
            d = np.load(p, mmap_mode='r')
            self.idx += [(p, k) for k in range(len(d['Y_t']))]
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        p, k = self.idx[i]
        if p not in self.cache:
            self.cache[p] = {k_: np.load(p, mmap_mode='r')[k_] for k_ in ('Y_t', 'A_t', 'Y_next', 'C')}
        c = self.cache[p]
        return (torch.from_numpy(np.asarray(c['Y_t'][k])), torch.from_numpy(np.asarray(c['A_t'][k])),
                torch.from_numpy(np.asarray(c['Y_next'][k])), torch.tensor(c['C'][k]))

import glob
ds = ShardDataset(sorted(glob.glob(f'{DATA}/*.npz')))
print('pairs:', len(ds), '| sample:', ds[0][0].shape, ds[0][1].shape)

## Path A — scaled action-conditioned surrogate (from scratch)

Same `(Y_t, a_t, C) → Y_{t+1}` residual task as `models/model_b.py`, wider/deeper.
Pick a config for your GPU: **S** (~10M, T4) / **M** (~100M, A100-40GB) /
**L** (~1B DiT, G4-96GB only). Param counter + memory estimate print first —
never train blind at this scale.

In [ ]:
import torch.nn as nn

class ScaledUNet(nn.Module):
    '''Action-conditioned residual UNet. base=width, depth=levels. Matches model_b I/O.'''
    def __init__(self, base=96, depth=3, in_ch=6):
        super().__init__()
        self.depth, chs = depth, []
        c = base
        self.enc, self.down = nn.ModuleList(), nn.ModuleList()
        prev = in_ch
        for d in range(depth):
            self.enc.append(nn.Sequential(nn.Conv2d(prev, c, 3, padding=1), nn.SiLU(),
                                            nn.Conv2d(c, c, 3, padding=1), nn.SiLU()))
            chs.append(c)
            if d < depth - 1:
                self.down.append(nn.Conv2d(c, 2 * c, 3, stride=2, padding=1))
            prev, c = 2 * c, 2 * c
        self.mid = nn.Sequential(nn.Conv2d(prev // 2, prev, 3, padding=1), nn.SiLU(),
                                  nn.Conv2d(prev, prev // 2, 3, padding=1), nn.SiLU())
        self.up, self.dec = nn.ModuleList(), nn.ModuleList()
        for d in reversed(range(depth - 1)):
            self.up.append(nn.ConvTranspose2d(chs[d + 1], chs[d], 4, stride=2, padding=1))
            self.dec.append(nn.Sequential(nn.Conv2d(2 * chs[d], chs[d], 3, padding=1), nn.SiLU(),
                                            nn.Conv2d(chs[d], chs[d], 3, padding=1), nn.SiLU()))
        self.out = nn.Conv2d(base, 2, 3, padding=1)
    def forward(self, x):
        feats, h = [], x
        for d in range(self.depth):
            h = self.enc[d](h); feats.append(h)
            if d < self.depth - 1: h = self.down[d](h)
        h = self.mid(h)
        for i, d in enumerate(reversed(range(self.depth - 1))):
            h = self.up[i](h)
            h = self.dec[i](torch.cat([h, feats[d]], 1))
        return self.out(h)

def count(m): return sum(p.numel() for p in m.parameters())
for name, (b, dp) in {'S-T4': (96, 3), 'M-A100': (192, 4)}.items():
    m = ScaledUNet(b, dp)
    n = count(m)
    print(f'{name}: {n/1e6:.1f}M params | bf16 {2*n/1e9:.1f}GB | adam-fp32 ~{12*n/1e9:.1f}GB | 8bit-adam ~{6*n/1e9:.1f}GB')
print('L-1B: use Path B (pretrained DiT + LoRA) unless on G4-96GB.')

In [ ]:
import math
from torch.utils.data import DataLoader
from virtual_lab.oracle import rasterize_force

CONFIG, BATCH, ACCUM, EPOCHS, LR = 'M-A100', 8, 8, 10, 2e-4  # eff batch 64. S-T4: CONFIG='S-T4', BATCH=8, ACCUM=2
BASE, DEPTH = {'S-T4': (96, 3), 'M-A100': (192, 4)}[CONFIG]
N = N_GRID
device = 'cuda'
use_bf16 = torch.cuda.is_bf16_supported()

def collate(batch):
    Yt = torch.stack([b[0] for b in batch])
    Yn = torch.stack([b[2] for b in batch])
    xs = []
    for Y, a, nu in [(b[0].numpy(), b[1].numpy(), float(b[3])) for b in batch]:
        dn, amp, ang, fx_, fy_, sig = a
        fxr, fyr = rasterize_force({'amp': float(amp), 'angle': float(ang), 'x': float(fx_), 'y': float(fy_), 'sigma': float(sig)}, N)
        xs.append(np.stack([Y[0], Y[1], fxr, fyr, np.full((N, N), nu * 20.0), np.zeros((N, N))], 0))
    return torch.from_numpy(np.stack(xs).astype(np.float32)), (torch.from_numpy(np.stack([b[2].numpy() for b in batch]).astype(np.float32)) - Yt)

net = ScaledUNet(BASE, DEPTH).to(device)
print(f'{CONFIG}: {count(net)/1e6:.1f}M params')
try:
    import bitsandbytes as bnb
    opt = bnb.optim.Adam8bit(net.parameters(), lr=LR)
    print('8-bit Adam')
except Exception:
    opt = torch.optim.AdamW(net.parameters(), lr=LR)
    print('AdamW fallback')
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS * math.ceil(len(ds) / BATCH / ACCUM))
scaler = torch.amp.GradScaler('cuda', enabled=use_bf16)
loss_fn = nn.MSELoss()
loader = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

net.train()
for ep in range(EPOCHS):
    tot = 0.0
    for it, (x, tgt) in enumerate(loader):
        x, tgt = x.to(device), tgt.to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16 if use_bf16 else torch.float16):
            loss = loss_fn(net(x.float()), tgt) / ACCUM
        scaler.scale(loss).backward()
        if (it + 1) % ACCUM == 0:
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        tot += loss.item() * ACCUM
    print(f'ep {ep}: loss {tot / len(loader):.3e}', flush=True)
    torch.save(net.state_dict(), f'{CKPT}/scaled_{CONFIG}_ep{ep}.pt')
print('Path A done ->', CKPT)

## Path B — 1B pretrained backbone + LoRA (recommended 1B route)

Full 1B training needs 80–96 GB VRAM. Instead: load a ~1B generative
backbone in 4-bit, attach LoRA adapters + a small action-conditioning head,
and fine-tune only the adapters on fluid `do(A)` pairs. Fits a T4.

In [ ]:
# Set your 1B backbone: any HF causal/diffusion-transformer ~1B with a transparent license.
BACKBONE = 'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'  # placeholder: swap for a 1B video/DiT backbone
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch.nn as nn

tok = AutoTokenizer.from_pretrained(BACKBONE)
base = AutoModel.from_pretrained(BACKBONE, load_in_4bit=True, device_map='auto',
                                   torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
base = prepare_model_for_kbit_training(base)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='FEATURE_EXTRACTION')
model = get_peft_model(base, lora)

class ActionHead(nn.Module):
    '''Maps (backbone pooled feat + 6-dim action vec) -> residual (du,dv) @256^2.'''
    def __init__(self, feat=2048, n=256):
        super().__init__()
        self.n = n
        self.mlp = nn.Sequential(nn.Linear(feat + 6, 1024), nn.SiLU(), nn.Linear(1024, 32 * 32 * 8))
        self.up = nn.Sequential(nn.ConvTranspose2d(8, 8, 4, 2, 1), nn.SiLU(),   # 32->64
                                nn.ConvTranspose2d(8, 4, 4, 2, 1), nn.SiLU(),   # 64->128
                                nn.ConvTranspose2d(4, 2, 4, 2, 1))               # 128->256
    def forward(self, feat, avec):
        h = self.mlp(torch.cat([feat, avec], -1)).view(-1, 8, 32, 32)
        return self.up(h)

model.print_trainable_parameters()  # expect <2% trainable: adapters + head only

In [ ]:
# Fine-tune loop (adapters + ActionHead). Patch-field encoding of Y_t left as exercise:
# encode each 256^2 field pair to backbone tokens, pool -> feat, concat action vec.
head = ActionHead().cuda().train()
opt = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=1e-4)
print('Train only LoRA adapters + ActionHead; backbone frozen 4-bit. See cell above for wiring.')
print('Save: model.save_pretrained(CKPT + \'/lora_1B\'); torch.save(head.state_dict(), CKPT + \'/action_head.pt\')')

## Evaluate — the intervention benchmark, not just loss

Same bar as the 110k model: T1 one-step, T2 rollout, T6 validated horizon
`H_eps`, T7 discovery + `Δ_V2R` / `R_discovery`. A 1B model that wins loss
but loses counterfactual ranking is a failure here.

In [ ]:
import json, numpy as np
from virtual_lab.lab import Lab
from virtual_lab.oracle import BurgersOracle
from virtual_lab.metrics_v2r import delta_v2r, r_discovery

# Wrap your trained net as a Lab surrogate: predict(state, action) -> state
# surrogate = YourWrapper(net, n=N_GRID)  # implement from Path A or B cells
# lab = Lab(BurgersOracle(n=N_GRID), surrogate=surrogate)
# s0 = lab.reset(seed=0, nu=0.01)
# acts = [{'d_nu': 0.0, 'force': {'amp': 1.0, 'angle': 0.2*i, 'x': 0.5, 'y': 0.5, 'sigma': 0.1}} for i in range(20)]
# print('H_eps(1e-3):', lab.validated_horizon(acts, eps=1e-3, init=s0))
print('Wire `surrogate` above, then run the T1–T7 suite from virtual_lab/benchmarks.py + agent_example.py.')

## Next

- 110k (repo) → S (~10M, T4) → M (~100M, A100) → 1B LoRA (any GPU) → 1B full (G4-96GB).
- Scale **data + resolution + action coverage** with params; re-measure T2/T6/T7 at each step.
- Results go back to `RESULTS.json`; only rank-preserving (`R_discovery`↑, `|Δ_V2R|`↓) scaling counts.